import ipywidgets as widgets
¿Qué hace?: Trae a nuestro programa una biblioteca llamada ipywidgets y le da el apodo de widgets para que sea más fácil de escribir.
¿Para qué sirve?: Esta biblioteca contiene todas las piezas visuales (botones, cuadros de texto, deslizadores) que usaremos para armar la aplicación.

from IPython.display import display
¿Qué hace?: Importa una función especial llamada display.
¿Para qué sirve?: Sirve para "dibujar" o mostrar los componentes visuales en la pantalla de tu cuaderno de notas (Colab)

widgets.FloatText(...): Crea una casilla especial para ingresar números decimales (en programación llamados float o punto flotante), como precios (ej. 10.50).
value=0.0: Empieza con el valor inicial de cero.

widgets.BoundedIntText(...): Crea una casilla numérica que solo acepta números enteros (sin decimales) y que además está "limitada" (bounded) dentro de un rango.
value=0: Empieza en cero.
min=0: No permite ingresar números menores a cero (evita stock negativo).
max=1000: No permite ingresar un número mayor a 1000.
step=1: Configura que el número suba o baje de 1 en 1 cuando usas las flechitas del teclado.


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display,HTML

datos = [
    {'codigo': 'A001', 'nombre': 'Arroz 1 kg', 'categoria': 'Abarrotes', 'precio': 1500, 'stock': 20, 'proveedor': 'Distribuidora Sur'},
    {'codigo': 'A002', 'nombre': 'Fideos 400 g', 'categoria': 'Abarrotes', 'precio': 900, 'stock': 35, 'proveedor': 'Distribuidora Sur'},
    {'codigo': 'B001', 'nombre': 'Leche entera 1 L', 'categoria': 'Lácteos', 'precio': 1200, 'stock': 18, 'proveedor': 'Lácteos Cordillera'},
    {'codigo': 'B002', 'nombre': 'Yogur natural', 'categoria': 'Lácteos', 'precio': 850, 'stock': 25, 'proveedor': 'Lácteos Cordillera'},
    {'codigo': 'C001', 'nombre': 'Bebida cola 1.5 L', 'categoria': 'Bebidas', 'precio': 1900, 'stock': 12, 'proveedor': 'Bebidas Centro'},
    {'codigo': 'C002', 'nombre': 'Agua mineral 1.5 L', 'categoria': 'Bebidas', 'precio': 1000, 'stock': 30, 'proveedor': 'Bebidas Centro'},
    {'codigo': 'D001', 'nombre': 'Jabón líquido', 'categoria': 'Limpieza', 'precio': 2500, 'stock': 10, 'proveedor': 'Higiene Total'},
    {'codigo': 'D002', 'nombre': 'Esponja', 'categoria': 'Limpieza', 'precio': 700, 'stock': 40, 'proveedor': 'Higiene Total'}
]

inventario = pd.DataFrame(datos)
inventario

,codigo,nombre,categoria,precio,stock,proveedor
0,A001,Arroz 1 kg,Abarrotes,1500,20,Distribuidora Sur
1,A002,Fideos 400 g,Abarrotes,900,35,Distribuidora Sur
2,B001,Leche entera 1 L,Lácteos,1200,18,Lácteos Cordillera
3,B002,Yogur natural,Lácteos,850,25,Lácteos Cordillera
4,C001,Bebida cola 1.5 L,Bebidas,1900,12,Bebidas Centro
5,C002,Agua mineral 1.5 L,Bebidas,1000,30,Bebidas Centro
6,D001,Jabón líquido,Limpieza,2500,10,Higiene Total
7,D002,Esponja,Limpieza,700,40,Higiene Total


In [ ]:
# --- Funciones de Lógica del Negocio ---
def registrar_producto(inventario, codigo, nombre, categoria, precio, stock, proveedor):
    producto = inventario['codigo'] == codigo

    if producto.sum() > 0:
        return 'El código ya existe.'
    elif precio <= 0:
        return 'El precio debe ser mayor que cero.'
    elif stock < 0:
        return 'El stock no puede ser negativo.'
    else:
        nuevo_producto = {
            'codigo': codigo,
            'nombre': nombre,
            'categoria': categoria,
            'precio': precio,
            'stock': stock,
            'proveedor': proveedor
        }
        inventario.loc[len(inventario)] = nuevo_producto
        return 'Producto registrado correctamente.'

def buscar_por_nombre(inventario, texto):
    texto = texto.lower()
    nombres = inventario['nombre'].str.lower()
    resultado = inventario[nombres.str.contains(texto)]
    return resultado

def mostrar_busqueda(evento):
    resultados = buscar_por_nombre(inventario, entrada_buscar.value)
    salida_busqueda.value = resultados.to_html()
    if not resultados.empty:
        producto_encontrado = resultados.iloc[0]
        codigo.value = str(producto_encontrado['codigo'])
        nombre.value = str(producto_encontrado['nombre'])
        categoria.value = str(producto_encontrado['categoria'])
        precio.value = float(producto_encontrado['precio'])
        proveedor.value = str(producto_encontrado['proveedor'])
        cantidad.value = str(producto_encontrado['stock'])
        salida.value = buscar_por_nombre(inventario, entrada_buscar.value).to_html()
    if resultados.empty:
        salida.value = "<p style='color: red;'>No se encontraron resultados.</p>"
def mostrar_productos(boton):
    if len(inventario) == 0:
        salida.value = "No hay productos registrados."
    else:
        salida.value = inventario.to_html()

def modificar_datos(boton):
    codigo_buscar = codigo.value.strip()
    coincidencia = inventario[inventario['codigo'] == codigo_buscar]

    if not coincidencia.empty:
        # Rellenamos los campos con los valores del DataFrame
        nombre.value = coincidencia.iloc[0]['nombre']
        categoria.value = coincidencia.iloc[0]['categoria']
        precio.value = float(coincidencia.iloc[0]['precio'])
        cantidad.value = int(coincidencia.iloc[0]['stock'])
        proveedor.value = coincidencia.iloc[0]['proveedor']
        salida.value = "<p style='color: blue;'>Datos cargados en el formulario. Modifícalos y presiona 'Modificar'.</p>"
    else:
        salida.value = "<p style='color: red;'>Código no encontrado en el inventario.</p>"

def actualizar_producto(boton):
    codigo_buscar = codigo.value.strip()
    indice = inventario[inventario['codigo'] == codigo_buscar].index

    if not indice.empty:
        # Reemplazamos los valores utilizando .loc en pandas
        inventario.loc[indice[0], 'nombre'] = nombre.value
        inventario.loc[indice[0], 'categoria'] = categoria.value
        inventario.loc[indice[0], 'precio'] = precio.value
        inventario.loc[indice[0], 'stock'] = cantidad.value
        inventario.loc[indice[0], 'proveedor'] = proveedor.value
        salida.value = "<p style='color: green; font-weight: bold;'>¡Producto actualizado con éxito!</p>"
    else:
        salida.value = "<p style='color: red; font-weight: bold;'>Error: No se pudo actualizar porque el código no existe.</p>"



In [ ]:
# --- Conexión de Eventos a los Botones ---

def al_registrar(boton):
    resultado = registrar_producto(
        inventario,
        codigo.value,
        nombre.value,
        categoria.value,
        precio.value,
        cantidad.value,
        proveedor.value
    )

    salida.value = resultado

def mostrar_productos(boton):
    salida.value = ""
    if inventario.empty:
        salida.value = "<p style='color: orange; font-weight: bold;'>No hay productos en el sistema.</p>"
    else:
        categoria_filtrada = categoria.value
        productos_filtrados = inventario[inventario['categoria'] == categoria_filtrada]

        if productos_filtrados.empty:
            salida.value = f"<p style='color: orange;'>No hay productos registrados en la categoría: <b>{categoria_filtrada}</b></p>"
        else:
            resultado_html = f"<h4 style='color: grey;'>Listado de Productos en Categoría ({categoria_filtrada}):</h4>"
            resultado_html += productos_filtrados.to_html(index=False)
            salida.value = resultado_html

def limpiar_campos(boton):
    nombre.value = ''
    categoria.value = 'Lácteos'
    codigo.value = ''
    proveedor.value = ''
    precio.value = 0.0
    cantidad.value = 0
    if boton is not None:
        salida.value = "<p style='color: blue;'>Campos del formulario limpiados.</p>"

def vaciar_lista(boton):
    global inventario
    inventario = inventario.iloc[0:0] # Vacía el DataFrame manteniendo la estructura
    salida_busqueda.clear_output() # Limpia la pantalla de búsqueda también
    salida.value = "<p style='color: red; font-weight: bold;'>La lista de productos ha sido vaciada por completo.</p>"


boton_buscar = widgets.Button(description='Buscar', button_style='primary')
boton_registrar = widgets.Button(description='Registrar Producto', button_style='success')
boton_mostrar = widgets.Button(description='Mostrar Productos', button_style='info')
boton_limpiar = widgets.Button(description='Limpiar Campos', button_style='warning')
boton_vaciar = widgets.Button(description='Vaciar Lista', button_style='danger')
boton_modificar=widgets.Button(description='Modificar Producto', button_style='warning')

boton_registrar.on_click(al_registrar)
boton_mostrar.on_click(mostrar_productos)
boton_limpiar.on_click(limpiar_campos)
boton_vaciar.on_click(vaciar_lista)
boton_buscar.on_click(mostrar_busqueda)
boton_modificar.on_click(actualizar_producto)


In [ ]:

# Inicialización de la Interfaz y Widgets
titulo = widgets.HTML(
    value="<h2 style='color: #2E5B88; font-family: sans-serif;'>Sistema de Administración de Minimarket</h2>"
)
# Elementos del Buscador Integrado (Justo después del formulario)
entrada_buscar = widgets.Text(placeholder='Ej: leche', description='Buscar:')
salida_busqueda = widgets.Output()

# Campos del Formulario de Registro
categoria = widgets.Dropdown(
    options=['Lácteos', 'Abarrotes', 'Limpieza', 'Bebidas', 'Snacks', 'Otros'],
    description='Categoría:')
nombre = widgets.Text(value='', placeholder='Ej:', description='Producto:')
codigo = widgets.Text(value='', placeholder='Codigo del producto', description='Codigo:')
proveedor = widgets.Text(value='', placeholder='Proveedor oficial', description='Proveedor:')
precio = widgets.FloatText(value=0.0, description='Precio Venta:')
cantidad = widgets.BoundedIntText(value=0, min=0, max=1000, step=1, description='Cantidad:')

# Área de salida para mostrar mensajes y la tabla general
salida = widgets.HTML(value="<p style='color: grey;'>No se han realizado acciones todavía.</p>")


In [ ]:
#Celda 5: Organización y visualización
widget_fila_0 = widgets.HBox([
    entrada_buscar,
    boton_buscar
])
widget_fila_1 = widgets.HBox([
    codigo,
    nombre
])

widget_fila_2 = widgets.HBox([
    categoria,
    proveedor
])

widget_fila_3 = widgets.HBox([
    precio,
    cantidad
])

widget_fila_botones = widgets.HBox([
    boton_registrar,
    boton_mostrar,
    boton_modificar,
    boton_limpiar,
    boton_vaciar
])

widget_formulario = widgets.VBox([
    widget_fila_0,
    salida_busqueda,
    widgets.HTML("<hr style='border: 0.5px solid #ccc;'>"),
    widgets.HTML("<h3>Registro de productos</h3>"),
    widget_fila_1,
    widget_fila_2,
    widget_fila_3,
    widget_fila_botones
])

widget_app = widgets.VBox([
    titulo,
    widget_formulario,
    widgets.HTML("<h3>Resultados</h3>"),
    salida
])

display(widget_app)